# Linux Mounting Block Devices and fstab (Educational Notebook)
This notebook covers how to attach ("mount") a filesystem into the directory tree, how to inspect and control mount options, and how to configure filesystems to mount automatically at boot via `/etc/fstab`.

## 1. What Does "Mounting" Mean?

On Linux there is a single, unified directory tree starting at `/` - there is no concept of separate drive letters like `C:` or `D:`. **Mounting** is the act of attaching a filesystem (from a partition, a USB drive, a network share, or an image file) onto an existing empty directory, called the **mount point**, making its contents appear at that location in the tree.

Before it's mounted, a filesystem's contents are inaccessible through the directory tree - the device exists (e.g. `/dev/sdb1`), but nothing under `/mnt/data` (say) will show its files until it's actually mounted there.


## 2. Manually Mounting a Filesystem

```bash
sudo mkdir -p /mnt/data                 # the mount point must already exist as a directory
sudo mount /dev/sdb1 /mnt/data             # mount a partition at that mount point
sudo mount -t ext4 /dev/sdb1 /mnt/data       # explicitly specify the filesystem type (usually auto-detected)
sudo mount -o ro /dev/sdb1 /mnt/data           # mount read-only

ls /mnt/data                                     # the partition's contents are now visible here
```

If you mount something onto a directory that already had files in it, those files aren't deleted - they're just hidden underneath the mount until you unmount again, at which point they reappear.


## 3. Viewing Mounted Filesystems

```bash
mount                          # list every currently mounted filesystem, with its options
findmnt                          # a cleaner, tree-structured view of mounts
findmnt /mnt/data                  # show details for one specific mount point
df -hT                                # show mounted filesystems, their type, and free/used space
cat /proc/mounts                       # the kernel's own live view of current mounts
```


## 4. Mount Options

Mount options control how the kernel treats a filesystem once it's mounted. Common ones:

| Option | Meaning |
|---|---|
| `ro` | Mount read-only |
| `rw` | Mount read-write (the default) |
| `noexec` | Prevent executing binaries located on this filesystem |
| `nosuid` | Ignore setuid/setgid bits on this filesystem |
| `nodev` | Don't interpret device files on this filesystem |
| `defaults` | A shorthand for the standard set of common options (`rw`, `suid`, `dev`, `exec`, `auto`, `nouser`, `async`) |

```bash
sudo mount -o noexec,nosuid /dev/sdb1 /mnt/usb      # multiple options, comma-separated, no spaces
sudo mount -o remount,rw /                             # change the options of an already-mounted filesystem in place
```

`noexec`, `nosuid`, and `nodev` are commonly applied together to removable media and other untrusted mount points, as a defense-in-depth measure against a plugged-in device trying to run code.


## 5. Unmounting

```bash
sudo umount /mnt/data           # unmount by mount point
sudo umount /dev/sdb1              # or unmount by device -- either works
```

Note the command is `umount`, not "unmount" - a common typo.

If unmounting fails with "target is busy", something still has an open file or is using that directory as its current working directory:

```bash
lsof +D /mnt/data          # list processes with open files under that mount point
fuser -vm /mnt/data          # a lighter-weight alternative that reports the same kind of information
sudo umount -l /mnt/data       # "lazy" unmount: detach immediately, actually free it once nothing is using it anymore
```


## 6. Mounting Removable Media and ISO Images

```bash
lsblk                                    # identify the device name of a newly-plugged-in USB drive
sudo mount /dev/sdc1 /mnt/usb              # mount it like any other partition
sudo umount /mnt/usb                         # always unmount before physically removing it, to flush pending writes

sudo mount -o loop image.iso /mnt/iso          # mount an ISO image file as if it were a block device, via a loop device
```

A **loop device** lets an ordinary file be treated as a block device, which is how you can mount a disk image without writing it to physical media first.


## 7. The `/etc/fstab` File

Manually running `mount` only lasts until the next reboot. `/etc/fstab` ("filesystem table") lists filesystems that should be mounted automatically at boot (or on demand), one per line, with six whitespace-separated fields:

```
<device>          <mount point>   <type>   <options>       <dump>   <pass>
UUID=1234-5678     /data             ext4     defaults          0        2
/dev/sdb2          swap                swap     defaults            0        0
```

| Field | Meaning |
|---|---|
| device | What to mount - a device path, `UUID=...`, or `LABEL=...` |
| mount point | Where to mount it (`swap` for swap space, which has no mount point) |
| type | Filesystem type (`ext4`, `xfs`, `swap`, `vfat`, ...) |
| options | Comma-separated mount options, as covered above |
| dump | Used by the legacy `dump` backup utility; `0` disables it (the near-universal setting today) |
| pass | `fsck` check order at boot: `0` = never check, `1` = check first (root filesystem only), `2` = check after (everything else) |


## 8. Identifying Devices Robustly in `fstab`

Device names like `/dev/sdb1` are assigned by **enumeration order** at boot, which can shift if a disk is added, removed, or a USB drive is plugged into a different port - `/dev/sdb1` today might be `/dev/sdc1` after a hardware change. `UUID=` (or `LABEL=`) references the filesystem itself, not its position, so the entry keeps working regardless of enumeration order.

```bash
blkid                    # find each filesystem's UUID and label to use in fstab
lsblk -f                   # a more compact view of the same information
```

Using `UUID=` in `/etc/fstab` is the standard, recommended practice on essentially all modern distributions for exactly this reason.


## 9. Testing `fstab` Changes Safely

A broken `/etc/fstab` entry can, in the worst case, prevent a system from booting to a usable state. Test changes carefully:

```bash
sudo mount -a          # attempt to mount everything listed in fstab that isn't already mounted, without rebooting
findmnt --verify           # (on systems that support it) validate fstab syntax and check for common mistakes
```

Running `mount -a` after every edit, and confirming there are no errors, catches typos before they turn into a boot problem. Keeping a backup copy of a working `/etc/fstab` before editing it is cheap insurance.


## 10. A Note on `nofail` and Non-Essential Mounts

For anything that might not always be present at boot time - a removable drive, an optional network share - add the `nofail` option so the boot process continues even if that particular entry can't be mounted, rather than dropping to an emergency shell:

```
UUID=abcd-1234   /mnt/backup   ext4   defaults,nofail   0   2
```

For genuinely automatic, on-demand mounting of removable or network filesystems (mounting only when actually accessed, and unmounting after a period of inactivity), a dedicated tool like `autofs` is typically a better fit than a static `fstab` entry.


## Hands-on

If you have a spare partition, USB drive, or virtual machine disk to practice on:

```bash
lsblk
sudo mkdir -p /mnt/test
sudo mount /dev/sdb1 /mnt/test
findmnt /mnt/test
df -hT /mnt/test
sudo umount /mnt/test

blkid /dev/sdb1
# add a line like this to /etc/fstab (edit with a text editor, as root):
# UUID=<the-uuid-you-found>   /mnt/test   ext4   defaults,nofail   0   2
sudo mount -a
findmnt /mnt/test
```


## Review Questions

1. What does "mounting" a filesystem actually do, conceptually, on a system with a single unified directory tree?
2. What happens to files that were already inside a directory if you mount another filesystem on top of it?
3. Name two commands you could use to see what's currently mounted, besides reading `/etc/fstab`.
4. What do the `noexec` and `nosuid` mount options protect against, and where would you typically apply them?
5. What does "target is busy" mean when unmounting fails, and what command helps you find the cause?
6. Why is `UUID=` generally preferred over a device path like `/dev/sdb1` in `/etc/fstab`?
7. List, in order, the six fields of an `/etc/fstab` line and what each one means.
8. What does the `nofail` option accomplish, and why would you want it on a removable drive's fstab entry but maybe not on your root filesystem's entry?
9. What command lets you test an `/etc/fstab` edit without rebooting, and why is that important to do before trusting the change?


# Cheat Sheet

```
Mount/unmount:
  mkdir -p /mnt/point
  mount /dev/sdX1 /mnt/point           mount -t <type> ... | mount -o ro,noexec ...
  umount /mnt/point | umount /dev/sdX1
  umount -l /mnt/point                    lazy unmount if busy
  mount -o loop image.iso /mnt/iso          mount a file as a block device

Inspect mounts:
  mount            findmnt            findmnt /mnt/point
  df -hT              cat /proc/mounts

Troubleshoot busy unmount:
  lsof +D /mnt/point       fuser -vm /mnt/point

fstab fields:
  <device>  <mountpoint>  <type>  <options>  <dump>  <pass>
  UUID=...    /data          ext4     defaults,nofail   0     2

Find UUIDs/labels:
  blkid       lsblk -f

Test fstab safely:
  mount -a
```
